# 🫁 Pulmonology Chest X-Ray Classification: Pipeline 2 (EfficientNetV2-S + Focal Loss)

## 📌 Overview
This notebook demonstrates **Pipeline 2 (Version 2)**, featuring advanced medical AI practices:
- **Classes:** `COVID-19`, `Normal`, `Lung Opacity`, `Viral Pneumonia`
- **Model:** `EfficientNetV2-S` (Fused-MBConv architecture for high accuracy and parameter efficiency)
- **Data Augmentation:** Medically-safe transformations (Rotation $\pm 7^\circ$, Translation $\pm 5\%$, Contrast/Brightness Jitter)
- **Loss Function:** `Focal Loss` (Class-Imbalance Aware with $\gamma = 2.0$)
- **Learning Rate Scheduler:** `CosineAnnealingLR` with `AdamW` Optimizer
- **Engine:** `BaseTrainer` with Multi-Metric Tracking (Accuracy, Loss, F1, Precision, Recall)


### Step 1: Import Dependencies & Setup

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import pandas as pd

# Add project root directory to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))

from core.models.pulmonology.pipeline_2.data_engineering import get_dataloaders
from core.models.pulmonology.pipeline_2.model_engineering import get_efficientnet_v2_s_model
from core.models.pulmonology.pipeline_2.loss_engineering import get_loss_criterion
from core.engine.base_trainer import BaseTrainer, create_experiment_dir, Logger

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"🚀 Active Pipeline 2 Device: {device.upper()}")

### Step 2: Medically-Safe Data Engineering & Class Weight Computation
We compute class weights dynamically based on sample inverse frequencies to feed into Focal Loss.

In [ ]:
splits_dir = '../../../dataset/splits/pulmonology/chest_xray'
images_dir = '../../../dataset/processed/pulmonology/chest_xray/images'
batch_size = 32

train_loader, val_loader, class_weights = get_dataloaders(splits_dir, images_dir, batch_size=batch_size)

print(f"Training Batches: {len(train_loader)}")
print(f"Validation Batches: {len(val_loader)}")
print(f"Computed Class Weights: {class_weights.tolist()}")

### Step 3: Model Engineering (EfficientNetV2-S)
EfficientNetV2-S utilizes Fused-MBConv layers for accelerated training and feature extraction.

In [ ]:
model = get_efficientnet_v2_s_model(num_classes=4, pretrained=True)
print(model.classifier)

### Step 4: Focal Loss Function & Cosine Annealing Scheduler
Focal Loss focuses hard-to-classify medical pathologies while downweighting easy background samples.

In [ ]:
criterion = get_loss_criterion(loss_type='focal', class_weights=class_weights, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

### Step 5: Training Execution & Experiment Tracking
Launch the training loop with full state checkpointing, progress tracking, and dynamic visualization curves.

In [ ]:
exp_dir = create_experiment_dir(module_name="pulmonology", model_name="EfficientNetV2_S", base_path="../../../core/models")

trainer = BaseTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    exp_dir=exp_dir,
    patience=5,
    accumulation_steps=1
)

# Fit the model for 10 epochs
# trainer.fit(num_epochs=10, resume=False)